In [ ]:
# Temel AI ve arayüz kütüphaneleri
!pip install torch torchvision torchaudio
!pip install diffusers transformers accelerate safetensors
!pip install gradio

# Upscale (RealESRGAN) kütüphaneleri
!pip install basicsr realesrgan facexlib gfpgan

In [ ]:
import torch
import gradio as gr
import numpy as np
from diffusers import UNet2DModel, DDPMPipeline, DDIMScheduler
from safetensors.torch import load_file
from PIL import Image, ImageEnhance

# model ve cihaz ayarları
config_path = "/content/super_tiny(4).json"
weight_path = "/content/super_tiny(4).safetensors"
device = "cuda" if torch.cuda.is_available() else "cpu"

def load_upsampler():
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    return RealESRGANer(
        scale=4,
        model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        model=model,
        tile=0, tile_pad=10, pre_pad=0,
        half=True if torch.cuda.is_available() else False,
        device=torch.device(device)
    )

try:
    config = UNet2DModel.load_config(config_path)
    unet = UNet2DModel.from_config(config).to(device, torch.float16 if device == "cuda" else torch.float32)
    unet.load_state_dict(load_file(weight_path))

    scheduler = DDIMScheduler(
        num_train_timesteps=1000,
        beta_schedule="squaredcos_cap_v2",
        prediction_type="epsilon",
        clip_sample=True,
        thresholding=True
    )
    pipe = DDPMPipeline(unet=unet, scheduler=scheduler).to(device, torch.float16 if device == "cuda" else torch.float32)
    upsampler = load_upsampler()
except Exception as e:
    print(f"Hata: {e}")

def generate(seed, steps, use_upscale):
    generator = torch.Generator(device=device).manual_seed(int(seed))

    with torch.no_grad():
        image = pipe(
            num_inference_steps=int(steps),
            generator=generator,
            output_type="pil"
        ).images[0]

    if use_upscale:
        # ara geçiş 128px
        image = image.resize((128, 128), resample=Image.LANCZOS)

        # ai yükseltme 256px
        img_input = np.array(image)
        output_upscaled, _ = upsampler.enhance(img_input, outscale=2)
        image = Image.fromarray(output_upscaled)

        # son dokunuşlar
        image = ImageEnhance.Sharpness(image).enhance(1.2)
        image = ImageEnhance.Contrast(image).enhance(1.05)

    return image

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("botanical diffusion recovery")

    with gr.Row():
        with gr.Column():
            seed = gr.Number(label="seed", value=42)
            steps = gr.Slider(50, 500, value=250, step=10, label="inference steps")
            use_upscale = gr.Checkbox(label="256px upscale", value=True)
            btn = gr.Button("üret", variant="primary")

        with gr.Column():
            output = gr.Image(label="sonuç")

    btn.click(fn=generate, inputs=[seed, steps, use_upscale], outputs=output)

demo.launch(share=True)